## Import packages

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import glob
import yaml
import adataviz as av
import pym3c
import os,sys
import anndata
import matplotlib.pylab as plt
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import matplotlib as mpl
mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Arial'

In [ ]:
os.chdir(os.path.expanduser("~/Projects/BG/clustering/100kb"))

In [ ]:
! pwd

## Parameters

In [ ]:
rootdir = "/home/x-wding2/Projects/BG/clustering/100kb"
obs_cols = ["Region", "donor", "tech", "MajorType", "Study", "mCGFrac", "mCHFrac"]
metadata_path="cell_metadata_with_annotation.tsv" #"annotations.tsv"
batch_col=['Study','donor']
# adata_path="~/Projects/BG/adata/BG.100kb-HiC.h5ad"
# adata_path="~/Projects/BG/adata/BG.25kb-HiC.h5ad"
adata_path="~/Projects/BG/adata/BG.HiC.fasthigashi-500K.h5ad"
last_round=2

In [ ]:
param_yaml = os.path.expanduser("~/Projects/Github/pym3c/pym3c/notebooks/clustering/BG_100kb.yaml")
with open(param_yaml) as f:
    params = yaml.load(f, Loader=yaml.FullLoader)
rootdir="./" #os.path.abspath(os.path.expanduser(params['rootdir']))
marker_path = params['marker_path']
marker_sheet = params['marker_sheet']
marker_level = params['marker_level']
max_depth = params['max_depth']
palette_path = params['palette_path']
figdir="figures"
if not os.path.exists(figdir):
    os.mkdir(figdir)
# fig_basename=os.path.join(figdir,"100kb.HiC")
fig_basename=os.path.join(figdir,"BG.HiC.fasthigashi-500K")
# fig_basename=os.path.join(figdir,"25kb.HiC")

if adata_path is None:
    adata_path=os.path.join(rootdir,"L1/data/L1.annotated.adata.h5ad")
print(adata_path)
print(metadata_path)

## Merge clustering_name_annot cell type annotation

In [ ]:
adata = anndata.read_h5ad(os.path.expanduser(adata_path))
adata

In [ ]:
if metadata_path is None:
    # concatenate all level annotations from the each level annotated adata
    for depth in range(1,last_round+1):
        clustering_name_annot=depth2level[depth]
        level=f"L{depth}"
        print(level,clustering_name_annot)
        infiles=glob.glob(os.path.join(rootdir,f'{level}/data/*.annotated.adata.h5ad'))
        print(infiles)
        D1={}
        D2={}
        for file in infiles:
            # print(file)
            prefixes=os.path.basename(file).split('.')[0].split('_')
            if len(prefixes)==1:
                parent=''
            else:
                parent=' '.join(prefixes[1:])
            # print(parent)
            D=anndata.read_h5ad(file,backed='r').obs[level].to_dict()
            D1.update(D)
            D=anndata.read_h5ad(file,backed='r').obs[clustering_name_annot].to_dict()
            D2.update(D)
        for col,D in zip([level,clustering_name_annot],[D1,D2]):
            adata.obs[col]=adata.obs_names.map(D)
    for i in range(2,last_round+1):
        print(i)
        adata.obs[depth2level[i]]=adata.obs.apply(lambda x:x[depth2level[i]] if not pd.isna(x[depth2level[i]]) else x[depth2level[i-1]],axis=1)
else:
    if metadata_path.endswith('.csv'):
        metadata=pd.read_csv(metadata_path,index_col=0)
    else:
        metadata=pd.read_csv(os.path.expanduser(metadata_path),sep='\t',index_col=0)
    name_mapping={'Group': {'B cells': 'B cell',
  'GPi Core': 'GPi Core GABA',
  'GPi Shell': 'GPi Shell GABA-Glut',
  'OT D1 ICj': 'OT ICj Granule',
  'SN EBF2 GABA': 'IP EBF2 GABA',
  'SN SEMA5A GABA': 'IP SEMA5A GABA',
  'STRd D1 Matrix MSN': 'STRd D1 Matrix SPN',
  'STRd D1 Striosome MSN': 'STRd D1 Striosome SPN',
  'STR D1D2 Hybrid MSN': 'STR D1D2 eSPN',
  'STRd D2 Matrix MSN': 'STRd D2 Matrix SPN',
  'STRd D2 StrioMat Hybrid MSN': 'STR D2 SMYD2-HTR7 SPN',
  'STRd D2 Striosome MSN': 'STRd D2 Striosome SPN',
  'STRv D1 MSN': 'STRv D1 SPN',
  'STRv D1 NUDAP MSN': 'STRv D1 NUDAP eSPN',
  'STRv D2 MSN': 'STRv D2 SPN',
  'T cells': 'T cell',
  'F M GATA3 GABA': 'RN GATA3 GABA',
  'ACx MEIS2 GABA': 'OB MEIS2 GABA'},
 'Subclass': {'OT Granular GABA': 'OT Granule GABA',
  'SN PAX7 GABA': 'IP PAX7 GABA',
  'STR D1 MSN': 'STR D1 SPN',
  'STR D2 MSN': 'STR D2 SPN',
  'STR Hybrid MSN': 'STR Eccentric SPN',
  'ACx MEIS2 GABA': 'OB MEIS2 GABA'}}
    for level in name_mapping:
        print(level)
        metadata[level]=metadata[level].replace(name_mapping[level])
    ovlp_cells=list(set(metadata.index.tolist()) & set(adata.obs_names))
    adata=adata[ovlp_cells,:]
    adata.obs=metadata.loc[adata.obs_names]
adata.obs

In [ ]:
adata.obs['Subclass'].unique()

In [ ]:
# keep_cells=adata.obs.loc[adata.obs[clustering_name_annot]!='doublet'].index.tolist()
# adata=adata[keep_cells,:]

In [ ]:
adata.strings_to_categoricals()

## Visualization

In [ ]:
keep_cells=adata.obs.loc[~ (adata.obs['Group'].isin(['doublet','Glut','Unk','Low quality cells']))].index.tolist() #'Low quality cells',
print(len(keep_cells))
adata=adata[keep_cells,:]
keep_cells=adata.obs.query("TotalCisContacts > 50000").index.tolist()
adata=adata[keep_cells,:]
adata.shape

### Plot annotated clusters

In [ ]:
# no text annotation
clustering_name_annot='Class'
# for coord_base in ['tsne','umap']:
coord_base="umap"
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=False,text_anno=None,
                  save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}.pdf",palette=palette_path,
               #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                  axis_format=None,figsize=(4,3.5),show_legend=True) #

In [ ]:
# no text annotation
clustering_name_annot='Subclass'
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=False,text_anno=None,
                  save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}.pdf",palette=palette_path,
               #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                  axis_format=None,figsize=(5,3.5),show_legend=True) #

In [ ]:
# with code annotation
clustering_name_annot='Subclass'
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=True,text_anno=clustering_name_annot,
                  save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}_with_code.pdf",palette=palette_path,
               #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                  text_kws = dict(bbox=dict(
                     boxstyle='round', #ellipse, round
                     edgecolor='white', fill=True, linewidth=0.6, alpha=0.6)),
                  axis_format=None,figsize=(5,3.5),rectangle_marker=True) #

In [ ]:
# no text annotation
clustering_name_annot='Region'
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=False,text_anno=None,
                  save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}.pdf",palette=palette_path,
               #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                  axis_format=None,figsize=(4,3.5),show_legend=True) #

In [ ]:
# no text annotation
clustering_name_annot='Group'
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=False,text_anno=None,
                save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}.pdf",palette=palette_path,
                legend_kws=dict(bbox_to_anchor=(1.05,0.94),ncol=2),
                axis_format=None,figsize=(5,3.5),show_legend=True) #

In [ ]:
# before and after batch correction
if 'fasthigashi' not in adata_path:
    basis='tsne'
    has_tsne_raw=True
    if "X_tsne_raw" not in adata.obsm:
        has_tsne_raw=False
    keys=list(pd.read_excel(os.path.expanduser(palette_path),sheet_name=None,index_col=0).keys())
    selected_cols=['Study','donor','Region']
    if not batch_col is None and has_tsne_raw and len(selected_cols)  > 0:
        basises=['tsne_raw','tsne'] # columns
        if len(selected_cols) > 0:
            nrows=len(selected_cols)
            ncols=len(basises)
            fig, axes = plt.subplots(nrows=nrows, ncols=ncols, 
                                     figsize=(ncols * 2.5, nrows * 2), dpi=300,
                                     constrained_layout=True,gridspec_kw=dict(wspace=0.1,hspace=0.1))
            if nrows==1:
                axes=axes.reshape(1,-1)
            for i,col in enumerate(selected_cols):
                if adata.obs[col].dropna().nunique()==0:
                    continue
                for j,basis in enumerate(basises):
                    ax=axes[i,j]
                    show_legend = True if (j + 1) == ncols else False
                    av.pl.plot_categorical(adata=adata,basis=basis,groupby=col,coding=False,text_anno=None,
                       save=None,palette=palette_path,show=False,ax=ax,
                    #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                       axis_format=None,figsize=(4,3.5),show_legend=show_legend) #
                    if i==0:
                        ax.set_title(basis, fontsize=10)
                    if j==0:
                        ax.set_ylabel(col, fontsize=10)
                    # ax.axis('off')
            plt.tight_layout()
            plt.savefig(f"{fig_basename}.{col}.batch_correction.clustering.pdf", dpi=300, bbox_inches='tight')  #
            plt.show()

In [ ]:
# av.pl.interactive_embedding(adata=adata,obs=None,variable='CellClass',gene=None,
#                   basis="umap",vmin='p1',vmax='p99',cmap='jet',annotation='Subclass',
#                   width=800,height=600,palette=palette_path,downsample=20000,size=5) # log2PlateNormCov

In [ ]:
# av.pl.interactive_embedding(adata=adata,obs=None,variable='Subclass',gene=None,
#                   basis="umap",vmin='p1',vmax='p99',cmap='jet',annotation='Region',
#                   width=800,height=600,palette=palette_path,downsample=20000,size=5) # log2PlateNormCov

In [ ]:
# av.pl.interactive_embedding(adata=adata,obs=None,variable='log2PlateNormCov',gene=None,
#                   basis="umap",vmin='p1',vmax='p99',cmap='jet',annotation='CellClass',
#                   width=1000,height=800,palette=palette_path,downsample=20000,size=5) # log2PlateNormCov

In [ ]:
# no text annotation
clustering_name_annot='donor'
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=False,text_anno=None,
                  save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}.pdf",palette=palette_path,
               #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                  axis_format=None,figsize=(4,3.5),show_legend=True) #

In [ ]:
# no text annotation
clustering_name_annot='Study'
av.pl.plot_categorical(adata=adata,basis=coord_base,groupby=clustering_name_annot,coding=False,text_anno=None,
                  save=f"{fig_basename}.annotated.{coord_base}.{clustering_name_annot}.pdf",palette=palette_path,
               #    legend_kws=dict(bbox_to_anchor=(0.1,0.9),ncol=2),
                  axis_format=None,figsize=(4,3.5),show_legend=True) #

In [ ]:
#PlateNormCov
fig, axes = plt.subplots(figsize=(9, 4), nrows=1,ncols=2, dpi=300)
ax=axes[0]
sc.pl.tsne(adata,color=['log2PlateNormCov'],
           ax=ax,
           wspace=0.25,ncols=2,show=False,
           vmin='p1',vmax='p99',frameon=False,
           cmap='jet' #vcenter='p50'
          )
colorbar = fig.axes[-1]
cur_pos=colorbar.get_position()
colorbar.set_position([cur_pos.x0,(1-cur_pos.height/2)/2,cur_pos.width, cur_pos.height / 2])

ax=axes[1]
sc.pl.umap(adata, #[adata.obs.PlateNormCov.between(0.5,1.5)],
           color=['log2PlateNormCov'],
           ax=ax,
           wspace=0.25,ncols=2,show=False,
           vmin='p1',vmax='p99',frameon=False,
           cmap='jet' #vcenter='p50'
          )
colorbar = fig.axes[-1]
cur_pos=colorbar.get_position()
colorbar.set_position([cur_pos.x0,(1-cur_pos.height/2)/2,cur_pos.width, cur_pos.height / 2])
plt.savefig(f"{fig_basename}.log2PlateNormCov.pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
plt.figure()
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (100)}):
    sc.pl.embedding(adata,basis='umap',color=['mCHFrac','mCGFrac','Study','is_doublet','Region'], #slab_distance
               wspace=0.25,ncols=3,show=False,frameon=False,
               use_raw=False, #legend_loc='on data',
               vmin='p5',vmax='p95',
               cmap='jet' #vcenter='p50'
              )
plt.savefig(f"{fig_basename}.umap.obs_cols.pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
obs1=av.tl.get_obs(adata)
obs1.set_index('cell',inplace=True)
obs1